# NOTEBOOK INVENTORY

This notebook was empty at inspection time. The inventory below reflects the assignment walkthrough in `Final Assignment/tutorial.ipynb` and a minimal reproducible validation setup for the telecom data.

## Summary

- Dataframes: `client`, `record`, `df`, `df_clean`, `X`, `y`, `X_train`, `X_test`, `y_train`, `y_test`, `model`
- Merged dataframe shape: `(100000, 100)`
- Modeling dataframe shape: `(100000, 99)`
- Target column: `churn`
- Modeling features: `98`
- Preprocessing: drop `Customer_ID`, label-encode object columns
- Feature engineering: no persistent engineered features in the modeling pipeline
- Split logic: 70/30 stratified train-test split with `random_state=42`
- Model: `XGBClassifier`


In [17]:
import pandas as pd
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

def resolve_project_dirs(start: Path | None = None):
    """Resolve the assignment and project roots from any notebook launch location."""
    start = (start or Path.cwd()).resolve()
    for current in (start, *start.parents):
        if current.name == "Final Assignment" and (current / "telecom").is_dir() and (current / "data").is_dir():
            final_assignment_dir = current
            project_root = current.parent
            break
        if (current / "Final Assignment").is_dir() and (current / "docs").is_dir():
            project_root = current
            final_assignment_dir = current / "Final Assignment"
            break
    else:
        raise FileNotFoundError("Could not locate the project root from the current working directory.")
    docs_root = project_root / "docs"
    telecom_dir = final_assignment_dir / "telecom"
    intermediate_dir = final_assignment_dir / "data" / "intermediate"
    eda_reports_dir = docs_root / "4-EDA-reports"
    return project_root, final_assignment_dir, docs_root, telecom_dir, intermediate_dir, eda_reports_dir

project_root, final_assignment_dir, docs_root, telecom_dir, intermediate_dir, eda_reports_dir = resolve_project_dirs()

base_dir = telecom_dir
client = pd.read_csv(base_dir / "Client.csv")
record = pd.read_csv(base_dir / "Record.csv")

# Keep the modeling dataframe name used in the reference notebook.
df = record.merge(client, on="Customer_ID", how="inner")
df_clean = df.drop(columns=["Customer_ID"])
X = df_clean.drop(columns=["churn"])
y = df_clean["churn"]


In [18]:
print(f"Modeling dataframe name: df_clean")
print(f"Target column: churn")
print(f"Current row/column count: {df_clean.shape}")
print(f"Feature count: {X.shape[1]}")


Modeling dataframe name: df_clean
Target column: churn
Current row/column count: (100000, 99)
Feature count: 98


# MISSING VALUE TREATMENT

This section treats missing values on the existing modeling dataframe `df_clean` without reloading the data or dropping any features.

In [19]:
import pandas as pd
from pathlib import Path

# Work on the existing modeling dataframe only.
missing_value_df = df_clean.copy()
feature_cols = [col for col in missing_value_df.columns if col != 'churn']
missing_features = [col for col in feature_cols if missing_value_df[col].isna().any()]

summary_rows = []
for col in feature_cols:
    missing_before = int(missing_value_df[col].isna().sum())
    indicator_created = missing_before > 0
    imputation_strategy = 'none'

    if indicator_created:
        indicator_col = f'{col}_missing_ind'
        missing_value_df[indicator_col] = missing_value_df[col].isna().astype(int)

        if pd.api.types.is_numeric_dtype(missing_value_df[col]):
            fill_value = missing_value_df[col].median()
            missing_value_df[col] = missing_value_df[col].fillna(fill_value)
            imputation_strategy = 'median'
        else:
            missing_value_df[col] = missing_value_df[col].fillna('MISSING')
            imputation_strategy = 'MISSING'

    missing_after = int(missing_value_df[col].isna().sum())
    summary_rows.append({
        'feature': col,
        'missing_before': missing_before,
        'missing_after': missing_after,
        'indicator_created': indicator_created,
        'imputation_strategy': imputation_strategy,
    })

missingness_summary_df = pd.DataFrame(summary_rows)

# Preserve the treated dataframe as the next modeling state.
df_missingness = missing_value_df

output_dir_docs = eda_reports_dir
output_dir_docs.mkdir(parents=True, exist_ok=True)
output_dir_data = intermediate_dir
output_dir_data.mkdir(parents=True, exist_ok=True)
output_dir_telecom = telecom_dir
output_dir_telecom.mkdir(parents=True, exist_ok=True)

snapshot_path = output_dir_data / 'E1_missingness.parquet'
summary_csv_path = output_dir_telecom / 'missingness_summary.csv'
report_path = output_dir_docs / 'missingness_features_report.md'

missingness_summary_df.to_csv(summary_csv_path, index=False)

# Pandas->pyarrow parquet can hit a notebook-specific extension-type cleanup bug.
# Writing the Arrow table directly avoids the failing unregister path.
snapshot_table = pa.Table.from_pandas(df_missingness, preserve_index=False)
pq.write_table(snapshot_table, snapshot_path)

missing_before_total = int(df_clean.isna().sum().sum())
missing_after_total = int(df_missingness.isna().sum().sum())
indicators_created = int(missingness_summary_df['indicator_created'].sum())
shape_before = df_clean.shape
shape_after = df_missingness.shape
snapshot_created = snapshot_path.exists()

report_lines = [
    '# Missing Value Treatment Report',
    '',
    '## Validation',
    '',
    f'- Missing count before treatment: {missing_before_total}',
    f'- Missing count after treatment: {missing_after_total}',
    f'- Indicators created: {indicators_created}',
    f'- Dataframe shape before: {shape_before}',
    f'- Dataframe shape after: {shape_after}',
    f'- E1_missingness.parquet created successfully: {snapshot_created}',
    '',
    '## Summary Table',
    '',
    missingness_summary_df.to_markdown(index=False),
]
report_path.write_text('\n'.join(report_lines))

print(f'Missing features treated: {len(missing_features)}')
print(f'Missing count before treatment: {missing_before_total}')
print(f'Missing count after treatment: {missing_after_total}')
print(f'Indicators created: {indicators_created}')
print(f'Shape before: {shape_before}')
print(f'Shape after: {shape_after}')
print(f'Snapshot created: {snapshot_created}')
print(f'Report: {report_path}')
print(f'Summary CSV: {summary_csv_path}')
print(f'Snapshot parquet: {snapshot_path}')


Missing features treated: 43
Missing count before treatment: 342969
Missing count after treatment: 0
Indicators created: 43
Shape before: (100000, 99)
Shape after: (100000, 142)
Snapshot created: True
Report: /Users/koushiksadhukhan/projects/gci-telecom-domain-ml-model/docs/4-EDA-reports/missingness_features_report.md
Summary CSV: /Users/koushiksadhukhan/projects/gci-telecom-domain-ml-model/Final Assignment/telecom/missingness_summary.csv
Snapshot parquet: /Users/koushiksadhukhan/projects/gci-telecom-domain-ml-model/Final Assignment/data/intermediate/E1_missingness.parquet


---

## 4.Feature Governance


In [20]:
import pandas as pd
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

# Governance classification for the current modeling dataframe.
# Safe features are stable customer/profile attributes.
# Conditional features are usable only if they are observed before the decision point.
# Unknown features are ambiguous codes or fields whose business meaning is not clear enough from EDA alone.

SAFE_FEATURES = [
    'months', 'uniqsubs', 'actvsubs', 'new_cell', 'crclscod', 'asl_flag',
    'prizm_social_one', 'area', 'dualband', 'refurb_new', 'hnd_price',
    'phones', 'models', 'hnd_webcap', 'truck', 'rv', 'ownrent', 'lor',
    'dwlltype', 'marital', 'adults', 'infobase', 'income', 'numbcars',
    'HHstatin', 'dwllsize', 'forgntvl', 'ethnic', 'kid0_2', 'kid3_5',
    'kid6_10', 'kid11_15', 'kid16_17', 'creditcd', 'eqpdays',
]

CONDITIONAL_FEATURES = [
    'rev_Mean', 'mou_Mean', 'totmrc_Mean', 'da_Mean', 'ovrmou_Mean',
    'ovrrev_Mean', 'vceovr_Mean', 'datovr_Mean', 'roam_Mean', 'change_mou',
    'change_rev', 'drop_vce_Mean', 'drop_dat_Mean', 'blck_vce_Mean',
    'blck_dat_Mean', 'unan_vce_Mean', 'unan_dat_Mean', 'plcd_vce_Mean',
    'plcd_dat_Mean', 'recv_vce_Mean', 'recv_sms_Mean', 'comp_vce_Mean',
    'comp_dat_Mean', 'custcare_Mean', 'ccrndmou_Mean', 'cc_mou_Mean',
    'inonemin_Mean', 'threeway_Mean', 'mou_cvce_Mean', 'mou_cdat_Mean',
    'mou_rvce_Mean', 'owylis_vce_Mean', 'mouowylisv_Mean', 'iwylis_vce_Mean',
    'mouiwylisv_Mean', 'peak_vce_Mean', 'peak_dat_Mean', 'mou_peav_Mean',
    'mou_pead_Mean', 'opk_vce_Mean', 'opk_dat_Mean', 'mou_opkv_Mean',
    'mou_opkd_Mean', 'drop_blk_Mean', 'attempt_Mean', 'complete_Mean',
    'callfwdv_Mean', 'callwait_Mean', 'totcalls', 'totmou', 'totrev',
    'adjrev', 'adjmou', 'adjqty', 'avgrev', 'avgmou', 'avgqty', 'avg3mou',
    'avg3qty', 'avg3rev', 'avg6mou', 'avg6qty', 'avg6rev',
]

UNKNOWN_FEATURES = []

feature_to_governance = {}
for feature in SAFE_FEATURES:
    feature_to_governance[feature] = ('safe', 'Static or customer profile attribute available before prediction.')
for feature in CONDITIONAL_FEATURES:
    feature_to_governance[feature] = ('conditional', 'Operational or usage signal that is useful only if measured before the decision point.')
for feature in UNKNOWN_FEATURES:
    feature_to_governance[feature] = ('unknown', 'Business meaning is ambiguous from EDA alone.')

rows = []
for feature in df_clean.columns:
    governance_type, reason = feature_to_governance.get(feature, ('unknown', 'Not clearly classifiable from EDA alone.'))
    rows.append({'feature': feature, 'governance_type': governance_type, 'reason': reason})

feature_governance_df = pd.DataFrame(rows)

# Preserve the current full dataframe state for traceability.
governance_snapshot_df = df_clean.copy()

safe_count = int((feature_governance_df['governance_type'] == 'safe').sum())
conditional_count = int((feature_governance_df['governance_type'] == 'conditional').sum())
unknown_count = int((feature_governance_df['governance_type'] == 'unknown').sum())

report_dir = eda_reports_dir
report_dir.mkdir(parents=True, exist_ok=True)
telecom_dir.mkdir(parents=True, exist_ok=True)
intermediate_dir.mkdir(parents=True, exist_ok=True)

report_path = report_dir / 'feature_governance.md'
report_lines = [
    '# Feature Governance',
    '',
    f'- SAFE_FEATURES: {safe_count}',
    f'- CONDITIONAL_FEATURES: {conditional_count}',
    f'- UNKNOWN_FEATURES: {unknown_count}',
    '',
    '## Governance Registry',
    '',
    feature_governance_df.to_markdown(index=False),
]
report_path.write_text('\n'.join(report_lines))

csv_path = telecom_dir / 'feature_governance.csv'
feature_governance_df.to_csv(csv_path, index=False)
parquet_path = intermediate_dir / 'E2_governance.parquet'
# Pandas->pyarrow parquet can fail when Arrow tries to inspect notebook-defined
# Python objects or dynamically executed source. Writing the Arrow table directly
# avoids that source-introspection path.
governance_snapshot_table = pa.Table.from_pandas(governance_snapshot_df, preserve_index=False)
pq.write_table(governance_snapshot_table, parquet_path)

print('SAFE_FEATURES:', safe_count)
print('CONDITIONAL_FEATURES:', conditional_count)
print('UNKNOWN_FEATURES:', unknown_count)
print(f'feature_governance.csv created: {csv_path.exists()}')
print(f'E2_governance.parquet created: {parquet_path.exists()}')
print(f'Report: {report_path}')


SAFE_FEATURES: 35
CONDITIONAL_FEATURES: 63
UNKNOWN_FEATURES: 1
feature_governance.csv created: True
E2_governance.parquet created: True
Report: /Users/koushiksadhukhan/projects/gci-telecom-domain-ml-model/docs/4-EDA-reports/feature_governance.md


In [21]:
print(feature_governance_df.head(10).to_string(index=False))
print('Total features:', len(feature_governance_df))
print('Governance counts:', feature_governance_df['governance_type'].value_counts().to_dict())


    feature governance_type                                                                                 reason
   rev_Mean     conditional Operational or usage signal that is useful only if measured before the decision point.
   mou_Mean     conditional Operational or usage signal that is useful only if measured before the decision point.
totmrc_Mean     conditional Operational or usage signal that is useful only if measured before the decision point.
    da_Mean     conditional Operational or usage signal that is useful only if measured before the decision point.
ovrmou_Mean     conditional Operational or usage signal that is useful only if measured before the decision point.
ovrrev_Mean     conditional Operational or usage signal that is useful only if measured before the decision point.
vceovr_Mean     conditional Operational or usage signal that is useful only if measured before the decision point.
datovr_Mean     conditional Operational or usage signal that is useful only if m

# SKEWNESS TREATMENT

This section identifies skewed numeric features in the current modeling dataframe and creates log-transformed versions using `np.log1p()` without overwriting the original columns.

In [22]:
import numpy as np
import pandas as pd
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

# Work from the current modeling dataframe without mutating the original columns.
skewness_df = df_clean.copy()
numeric_features = [
    col for col in skewness_df.columns
    if col != 'churn' and pd.api.types.is_numeric_dtype(skewness_df[col])
]

skew_rows = []
for feature in numeric_features:
    original_skew = float(skewness_df[feature].skew())
    if abs(original_skew) > 1:
        transformed_col = f'{feature}_log'
        # Shift only when required so log1p stays within its numeric domain.
        min_value = float(skewness_df[feature].min())
        shift = 0.0 if min_value >= 0 else abs(min_value) + 1.0
        skewness_df[transformed_col] = np.log1p(skewness_df[feature] + shift)
        transformed_skew = float(skewness_df[transformed_col].skew())
        skew_rows.append({
            'feature': feature,
            'original_skew': original_skew,
            'transformed_skew': transformed_skew,
            'skew_reduction': abs(original_skew) - abs(transformed_skew),
        })

skewness_comparison_df = pd.DataFrame(skew_rows).sort_values(
    by=['skew_reduction', 'feature'], ascending=[False, True]
).reset_index(drop=True)

docs_dir = eda_reports_dir
telecom_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)
intermediate_dir.mkdir(parents=True, exist_ok=True)

report_path = docs_dir / 'skew_treatment_report.md'
report_lines = [
    '# SKEWNESS TREATMENT',
    '',
    'Numeric features with `abs(skewness) > 1` were transformed with `np.log1p()` to reduce skewness. Features with negative minima were shifted by a constant before transformation to keep the input domain valid.',
    '',
    '## Summary Table',
    '',
    skewness_comparison_df.to_markdown(index=False),
]
report_path.write_text('\n'.join(report_lines))

csv_path = telecom_dir / 'skewness_comparison.csv'
skewness_comparison_df.to_csv(csv_path, index=False)

parquet_path = intermediate_dir / 'E3_skewness.parquet'
skewness_snapshot_table = pa.Table.from_pandas(skewness_df, preserve_index=False)
pq.write_table(skewness_snapshot_table, parquet_path)

print(skewness_comparison_df.head(20).to_string(index=False))
print(f'skewness_comparison.csv created: {csv_path.exists()}')
print(f'E3_skewness.parquet created: {parquet_path.exists()}')


      feature  original_skew  transformed_skew  skew_reduction
blck_dat_Mean     224.599850         21.126060      203.473789
    roam_Mean     168.807785          2.995734      165.812051
recv_sms_Mean     168.439203         19.947221      148.491982
drop_dat_Mean     149.640425         11.973458      137.666967
unan_dat_Mean      81.813561         12.295313       69.518248
mou_opkd_Mean      67.435409          5.912650       61.522759
     uniqsubs      60.519797          1.359097       59.160701
callfwdv_Mean      90.737800         33.949265       56.788534
  datovr_Mean      58.007872          6.494296       51.513576
mou_cdat_Mean      49.327627          5.075516       44.252110
   change_rev      80.535420        -36.917324       43.618096
mou_pead_Mean      43.003812          6.119241       36.884571
custcare_Mean      32.032501          1.529393       30.503109
plcd_dat_Mean      30.273577          5.213441       25.060135
 opk_dat_Mean      29.966403          6.407814       23

In [23]:
top_20_skew_reductions = skewness_comparison_df.nlargest(20, 'skew_reduction')
print(top_20_skew_reductions.to_string(index=False))
print(f"skewness_comparison.csv created successfully: {(telecom_dir / 'skewness_comparison.csv').exists()}")
print(f"E3_skewness.parquet created successfully: {(intermediate_dir / 'E3_skewness.parquet').exists()}")


      feature  original_skew  transformed_skew  skew_reduction
blck_dat_Mean     224.599850         21.126060      203.473789
    roam_Mean     168.807785          2.995734      165.812051
recv_sms_Mean     168.439203         19.947221      148.491982
drop_dat_Mean     149.640425         11.973458      137.666967
unan_dat_Mean      81.813561         12.295313       69.518248
mou_opkd_Mean      67.435409          5.912650       61.522759
     uniqsubs      60.519797          1.359097       59.160701
callfwdv_Mean      90.737800         33.949265       56.788534
  datovr_Mean      58.007872          6.494296       51.513576
mou_cdat_Mean      49.327627          5.075516       44.252110
   change_rev      80.535420        -36.917324       43.618096
mou_pead_Mean      43.003812          6.119241       36.884571
custcare_Mean      32.032501          1.529393       30.503109
plcd_dat_Mean      30.273577          5.213441       25.060135
 opk_dat_Mean      29.966403          6.407814       23

# OUTLIER TREATMENT

Winsorize the prioritized numeric features at the 1st and 99th percentiles without overwriting the original columns.


In [24]:
import pandas as pd
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

outlier_features = [
    "change_rev",
    "change_mou",
    "roam_Mean",
    "ovrmou_Mean",
    "vceovr_Mean",
]

outlier_treated_df = df_clean.copy()
outlier_rows = []

for feature in outlier_features:
    lower = outlier_treated_df[feature].quantile(0.01)
    upper = outlier_treated_df[feature].quantile(0.99)
    capped_col = f"{feature}_capped"
    outlier_treated_df[capped_col] = outlier_treated_df[feature].clip(lower=lower, upper=upper)
    affected_rows = int(((outlier_treated_df[feature] < lower) | (outlier_treated_df[feature] > upper)).sum())
    outlier_rows.append({
        "feature": feature,
        "lower_bound": lower,
        "upper_bound": upper,
        "affected_rows": affected_rows,
        "original_min": outlier_treated_df[feature].min(),
        "original_max": outlier_treated_df[feature].max(),
        "capped_min": outlier_treated_df[capped_col].min(),
        "capped_max": outlier_treated_df[capped_col].max(),
        "capped_column": capped_col,
    })

outlier_summary_df = pd.DataFrame(outlier_rows).sort_values(
    by=["affected_rows", "feature"], ascending=[False, True]
).reset_index(drop=True)

docs_dir = eda_reports_dir
telecom_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)
intermediate_dir.mkdir(parents=True, exist_ok=True)

report_path = docs_dir / "outlier_treatment_report.md"
report_lines = [
    "# OUTLIER TREATMENT",
    "",
    "Winsorization at the 1st and 99th percentiles was applied to the prioritized features without changing the original columns.",
    "",
    "## Cap Summary",
    "",
    outlier_summary_df.to_string(index=False),
]
report_path.write_text("\n".join(report_lines), encoding="utf-8")

csv_path = telecom_dir / "outlier_summary.csv"
outlier_summary_df.to_csv(csv_path, index=False)

parquet_path = intermediate_dir / "E4_outlier.parquet"
pq.write_table(pa.Table.from_pandas(outlier_treated_df, preserve_index=False), parquet_path)

print("OUTLIER_FEATURES:", len(outlier_features))
print("outlier_summary.csv created successfully:", csv_path.exists())
print("E4_outlier.parquet created successfully:", parquet_path.exists())
print("Report created:", report_path.exists())


OUTLIER_FEATURES: 5
outlier_summary.csv created successfully: True
E4_outlier.parquet created successfully: True
Report created: True


In [25]:
print(outlier_summary_df[["feature", "lower_bound", "upper_bound", "affected_rows"]].to_string(index=False))
print("Cap thresholds and affected rows shown above.")
print("outlier_summary.csv created successfully:", (telecom_dir / "outlier_summary.csv").exists())
print("E4_outlier.parquet created successfully:", (intermediate_dir / "E4_outlier.parquet").exists())


    feature  lower_bound  upper_bound  affected_rows
 change_mou    -835.9800    751.46000           1984
 change_rev    -104.5743    121.88250           1984
  roam_Mean       0.0000     22.40935            997
vceovr_Mean       0.0000    137.76850            997
ovrmou_Mean       0.0000    430.75000            996
Cap thresholds and affected rows shown above.
outlier_summary.csv created successfully: True
E4_outlier.parquet created successfully: True


# EDA ACTION MATRIX

This section consolidates missingness, governance, skewness, and outlier outputs into a feature-level action plan.


In [26]:
import pandas as pd
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

def resolve_project_dirs(start: Path | None = None):
    """Resolve the assignment and project roots from any notebook launch location."""
    start = (start or Path.cwd()).resolve()
    for current in (start, *start.parents):
        if current.name == "Final Assignment" and (current / "telecom").is_dir() and (current / "data").is_dir():
            final_assignment_dir = current
            project_root = current.parent
            break
        if (current / "Final Assignment").is_dir() and (current / "docs").is_dir():
            project_root = current
            final_assignment_dir = current / "Final Assignment"
            break
    else:
        raise FileNotFoundError("Could not locate the project root from the current working directory.")
    docs_root = project_root / "docs"
    telecom_dir = final_assignment_dir / "telecom"
    intermediate_dir = final_assignment_dir / "data" / "intermediate"
    eda_reports_dir = docs_root / "4-EDA-reports"
    return project_root, final_assignment_dir, docs_root, telecom_dir, intermediate_dir, eda_reports_dir

if "telecom_dir" not in globals() or "intermediate_dir" not in globals() or "eda_reports_dir" not in globals():
    project_root, final_assignment_dir, docs_root, telecom_dir, intermediate_dir, eda_reports_dir = resolve_project_dirs()

# Reuse existing notebook outputs when available; otherwise load the saved artifacts.
if "df_clean" not in globals():
    client = pd.read_csv(telecom_dir / "Client.csv")
    record = pd.read_csv(telecom_dir / "Record.csv")
    df_clean = record.merge(client, on="Customer_ID", how="inner").drop(columns=["Customer_ID"])

if "missingness_summary_df" not in globals():
    missingness_summary_df = pd.read_csv(telecom_dir / "missingness_summary.csv")
if "feature_governance_df" not in globals():
    feature_governance_df = pd.read_csv(telecom_dir / "feature_governance.csv")
if "skewness_comparison_df" not in globals():
    skewness_comparison_df = pd.read_csv(telecom_dir / "skewness_comparison.csv")
if "outlier_summary_df" not in globals():
    outlier_summary_df = pd.read_csv(telecom_dir / "outlier_summary.csv")

eda_feature_cols = [col for col in df_clean.columns if col != "churn"]
missing_map = missingness_summary_df.set_index("feature").to_dict("index")
governance_map = feature_governance_df.set_index("feature")["governance_type"].to_dict()
skew_features = set(skewness_comparison_df["feature"].astype(str))
outlier_features = set(outlier_summary_df["feature"].astype(str))

action_rows = []
for feature in eda_feature_cols:
    missing_info = missing_map.get(feature, {})
    missing_before = int(missing_info.get("missing_before", 0))
    imputation_strategy = str(missing_info.get("imputation_strategy", "none"))
    if missing_before > 0:
        if imputation_strategy == "median":
            missing_strategy = "median + indicator"
        elif imputation_strategy == "MISSING":
            missing_strategy = "MISSING + indicator"
        else:
            missing_strategy = f"{imputation_strategy} + indicator"
    else:
        missing_strategy = "none"

    governance_type = governance_map.get(feature, "unknown")
    if governance_type == "unknown":
        encoding_strategy = "manual review"
        keep_candidate = False
    elif pd.api.types.is_numeric_dtype(df_clean[feature]):
        encoding_strategy = "none"
        keep_candidate = True
    else:
        cardinality = int(df_clean[feature].nunique(dropna=True))
        if cardinality <= 2:
            encoding_strategy = "label-encode"
        elif cardinality <= 10:
            encoding_strategy = "one-hot"
        else:
            encoding_strategy = "label-encode"
        keep_candidate = True

    in_skew = feature in skew_features
    in_outlier = feature in outlier_features
    if in_skew and in_outlier:
        transformation_strategy = "winsorize_1_99 + log1p"
    elif in_outlier:
        transformation_strategy = "winsorize_1_99"
    elif in_skew:
        transformation_strategy = "log1p"
    else:
        transformation_strategy = "none"

    action_rows.append({
        "feature": feature,
        "missing_strategy": missing_strategy,
        "transformation_strategy": transformation_strategy,
        "encoding_strategy": encoding_strategy,
        "governance_type": governance_type,
        "keep_candidate": keep_candidate,
    })

eda_action_matrix = pd.DataFrame(action_rows)
eda_action_matrix = eda_action_matrix.sort_values("feature").reset_index(drop=True)

telecom_dir.mkdir(parents=True, exist_ok=True)
docs_dir = eda_reports_dir
docs_dir.mkdir(parents=True, exist_ok=True)
intermediate_dir.mkdir(parents=True, exist_ok=True)

report_path = docs_dir / "eda_action_matrix.md"
report_lines = [
    "# EDA ACTION MATRIX",
    "",
    f"Rows: {len(eda_action_matrix)}",
    f"Features covered: {len(eda_feature_cols)}",
    f"Keep candidates: {int(eda_action_matrix['keep_candidate'].sum())}",
    "",
    "## Action Table",
    "",
    eda_action_matrix.to_string(index=False),
]
report_path.write_text("\n".join(report_lines), encoding="utf-8")

csv_path = telecom_dir / "eda_action_matrix.csv"
eda_action_matrix.to_csv(csv_path, index=False)

parquet_path = intermediate_dir / "E5_action_matrix.parquet"
pq.write_table(pa.Table.from_pandas(eda_action_matrix, preserve_index=False), parquet_path)

print("EDA_ACTION_ROWS:", len(eda_action_matrix))
print("one row per feature:", len(eda_action_matrix) == len(eda_feature_cols))
print("eda_action_matrix.csv created successfully:", csv_path.exists())
print("E5_action_matrix.parquet created successfully:", parquet_path.exists())
print("Report created:", report_path.exists())


EDA_ACTION_ROWS: 98
one row per feature: True
eda_action_matrix.csv created successfully: True
E5_action_matrix.parquet created successfully: True
Report created: True


In [27]:
print(eda_action_matrix.head(20).to_string(index=False))
print("one row per feature:", len(eda_action_matrix) == len([col for col in df_clean.columns if col != "churn"]))
print("eda_action_matrix.csv created successfully:", (telecom_dir / "eda_action_matrix.csv").exists())
print("E5_action_matrix.parquet created successfully:", (intermediate_dir / "E5_action_matrix.parquet").exists())


      feature    missing_strategy transformation_strategy encoding_strategy governance_type  keep_candidate
     HHstatin MISSING + indicator                    none           one-hot            safe            True
     actvsubs                none                   log1p              none            safe            True
       adjmou                none                   log1p              none     conditional            True
       adjqty                none                   log1p              none     conditional            True
       adjrev                none                   log1p              none     conditional            True
       adults  median + indicator                    none              none            safe            True
         area MISSING + indicator                    none      label-encode            safe            True
     asl_flag                none                    none      label-encode            safe            True
 attempt_Mean               

# CORRELATION CLUSTERS

This section groups numeric features into correlation clusters using an absolute correlation threshold of 0.80. The dataframe is retained in full; no columns are dropped yet.


In [28]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
import pyarrow as pa
import pyarrow.parquet as pq

CORR_THRESHOLD = 0.80


def resolve_project_dirs(start: Path | None = None):
    """Resolve the assignment and project roots from any notebook launch location."""
    start = (start or Path.cwd()).resolve()
    for current in (start, *start.parents):
        if current.name == "Final Assignment" and (current / "telecom").is_dir() and (current / "data").is_dir():
            final_assignment_dir = current
            project_root = current.parent
            break
        if (current / "Final Assignment").is_dir() and (current / "docs").is_dir():
            project_root = current
            final_assignment_dir = current / "Final Assignment"
            break
    else:
        raise FileNotFoundError("Could not locate the project root from the current working directory.")
    docs_root = project_root / "docs"
    telecom_dir = final_assignment_dir / "telecom"
    intermediate_dir = final_assignment_dir / "data" / "intermediate"
    eda_reports_dir = docs_root / "4-EDA-reports"
    return project_root, final_assignment_dir, docs_root, telecom_dir, intermediate_dir, eda_reports_dir

if "telecom_dir" not in globals() or "intermediate_dir" not in globals() or "eda_reports_dir" not in globals():
    project_root, final_assignment_dir, docs_root, telecom_dir, intermediate_dir, eda_reports_dir = resolve_project_dirs()

if "df_clean" not in globals():
    client = pd.read_csv(telecom_dir / "Client.csv")
    record = pd.read_csv(telecom_dir / "Record.csv")
    df_clean = record.merge(client, on="Customer_ID", how="inner").drop(columns=["Customer_ID"])

corr_feature_cols = [col for col in df_clean.columns if col != "churn" and pd.api.types.is_numeric_dtype(df_clean[col])]
abs_corr = df_clean[corr_feature_cols].corr().abs()

adjacency = defaultdict(set)
for i, left in enumerate(corr_feature_cols):
    for right in corr_feature_cols[i + 1:]:
        value = abs_corr.loc[left, right]
        if pd.notna(value) and value > CORR_THRESHOLD:
            adjacency[left].add(right)
            adjacency[right].add(left)

components = []
visited = set()
for feature in corr_feature_cols:
    if feature in visited:
        continue
    stack = [feature]
    visited.add(feature)
    component = []
    while stack:
        current = stack.pop()
        component.append(current)
        for neighbor in adjacency[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                stack.append(neighbor)
    if len(component) > 1:
        components.append(sorted(component))

components = sorted(components, key=lambda comp: (-len(comp), comp[0]))

usage_anchor = {
    "mou_Mean", "avgmou", "avg3mou", "avg6mou", "avgqty", "avg3qty", "avg6qty",
    "plcd_vce_Mean", "comp_vce_Mean", "attempt_Mean", "complete_Mean", "recv_vce_Mean",
    "onan_vce_Mean", "unan_vce_Mean", "owylis_vce_Mean", "mouowylisv_Mean",
    "mou_rvce_Mean", "mou_cvce_Mean", "mou_opkv_Mean", "mou_peav_Mean", "peak_vce_Mean",
    "inonemin_Mean",
}
revenue_anchor = {"totrev", "adjrev", "totmou", "adjmou", "adjqty", "totcalls"}
support_anchor = {"custcare_Mean", "ccrndmou_Mean", "cc_mou_Mean"}
service_anchor = {"plcd_dat_Mean", "comp_dat_Mean", "peak_dat_Mean", "opk_dat_Mean", "mou_cdat_Mean", "mou_opkd_Mean"}
quality_anchor = {"blck_vce_Mean", "drop_blk_Mean"}
overage_anchor = {"ovrmou_Mean", "ovrrev_Mean", "vceovr_Mean"}
device_anchor = {"phones", "models"}
data_service_anchor = {"mou_cdat_Mean", "mou_opkd_Mean"}
revenue_aux_anchor = {"rev_Mean", "avg3rev", "avg6rev", "avgrev"}

primary_representatives = {
    "usage": "mou_Mean",
    "revenue": "totrev",
    "support": "custcare_Mean",
    "service": "plcd_dat_Mean",
}

rows = []
for cluster_id, component in enumerate(components, start=1):
    component = sorted(component)
    sub_corr = abs_corr.loc[component, component]
    if len(component) == 2:
        max_pair_corr = float(sub_corr.iloc[0, 1])
    else:
        max_pair_corr = max(
            float(sub_corr.loc[left, right])
            for i, left in enumerate(component)
            for right in component[i + 1:]
        )

    centrality_scores = {}
    for feature in component:
        if len(component) == 1:
            centrality_scores[feature] = 1.0
        else:
            centrality_scores[feature] = float((sub_corr.loc[feature].sum() - 1.0) / (len(component) - 1))
    central_rep = max(centrality_scores, key=centrality_scores.get)

    theme = "other"
    representative = central_rep
    selection_role = "secondary"
    if set(component) & support_anchor:
        theme = "support"
        representative = primary_representatives[theme]
        selection_role = "primary"
    elif set(component) & revenue_anchor and len(component) >= 5:
        theme = "revenue"
        representative = primary_representatives[theme]
        selection_role = "primary"
    elif set(component) & usage_anchor and len(component) >= 10:
        theme = "usage"
        representative = primary_representatives[theme]
        selection_role = "primary"
    elif set(component) & service_anchor:
        theme = "service"
        representative = primary_representatives[theme]
        selection_role = "primary"
    elif set(component) & revenue_aux_anchor:
        theme = "revenue_aux"
        representative = "rev_Mean"
    elif set(component) & overage_anchor:
        theme = "overage"
        representative = "ovrrev_Mean"
    elif set(component) & quality_anchor:
        theme = "quality"
        representative = "blck_vce_Mean"
    elif set(component) & device_anchor:
        theme = "device"
        representative = "phones"
    elif set(component) & data_service_anchor:
        theme = "data_service"
        representative = "mou_cdat_Mean"

    rows.append({
        "cluster_id": cluster_id,
        "cluster_theme": theme,
        "selection_role": selection_role,
        "cluster_size": len(component),
        "representative_feature": representative,
        "central_feature": central_rep,
        "max_abs_pair_corr": max_pair_corr,
        "cluster_members": "; ".join(component),
    })

correlation_cluster_df = pd.DataFrame(rows).sort_values(
    by=["selection_role", "cluster_size", "cluster_id"],
    ascending=[True, False, True],
).reset_index(drop=True)

correlation_snapshot_df = df_clean.copy()

telecom_dir.mkdir(parents=True, exist_ok=True)
eda_reports_dir.mkdir(parents=True, exist_ok=True)
intermediate_dir.mkdir(parents=True, exist_ok=True)

report_path = eda_reports_dir / "correlation_cluster_report.md"
primary_cluster_df = correlation_cluster_df[correlation_cluster_df["selection_role"] == "primary"].copy()

report_lines = [
    "# CORRELATION CLUSTERS",
    "",
    f"Threshold: |corr| > {CORR_THRESHOLD:.2f}",
    f"Numeric features analyzed: {len(corr_feature_cols)}",
    f"Correlation clusters found: {len(correlation_cluster_df)}",
    "",
    "## Primary clusters",
    "",
    primary_cluster_df[["cluster_theme", "cluster_size", "representative_feature", "cluster_members"]].to_string(index=False),
    "",
    "## Full cluster table",
    "",
    correlation_cluster_df.to_string(index=False),
    "",
    "## Notes",
    "",
    "- The dataframe is kept intact; this step only identifies correlated groups.",
    "- Representative features are selected for the four requested business themes: usage, revenue, support, and service.",
]
report_path.write_text("\n".join(report_lines), encoding="utf-8")

csv_path = telecom_dir / "correlation_cluster.csv"
correlation_cluster_df.to_csv(csv_path, index=False)

parquet_path = intermediate_dir / "E6_correlation.parquet"
pq.write_table(pa.Table.from_pandas(correlation_snapshot_df, preserve_index=False), parquet_path)

print("CORRELATION_CLUSTERS:", len(correlation_cluster_df))
print("primary cluster summary:")
print(primary_cluster_df[["cluster_theme", "cluster_size", "representative_feature"]].to_string(index=False))
print("correlation_cluster.csv created successfully:", csv_path.exists())
print("E6_correlation.parquet created successfully:", parquet_path.exists())
print("report created:", report_path.exists())


CORRELATION_CLUSTERS: 9
primary cluster summary:
cluster_theme  cluster_size representative_feature
        usage            22               mou_Mean
      revenue             6                 totrev
      service             4          plcd_dat_Mean
      support             3          custcare_Mean
      service             2          plcd_dat_Mean
correlation_cluster.csv created successfully: True
E6_correlation.parquet created successfully: True
report created: True


In [29]:
print(correlation_cluster_df[["cluster_theme", "cluster_size", "representative_feature"]].to_string(index=False))
print("primary cluster count:", int((correlation_cluster_df["selection_role"] == "primary").sum()))
print("correlation_cluster.csv created successfully:", (telecom_dir / "correlation_cluster.csv").exists())
print("E6_correlation.parquet created successfully:", (intermediate_dir / "E6_correlation.parquet").exists())


cluster_theme  cluster_size representative_feature
        usage            22               mou_Mean
      revenue             6                 totrev
      service             4          plcd_dat_Mean
      support             3          custcare_Mean
      service             2          plcd_dat_Mean
  revenue_aux             4               rev_Mean
      overage             3            ovrrev_Mean
      quality             2          blck_vce_Mean
       device             2                 phones
primary cluster count: 5
correlation_cluster.csv created successfully: True
E6_correlation.parquet created successfully: True


# FEATURE ENGINEERING

This section adds a compact set of derived telecom signals for usage, revenue, trend, service issues, and support intensity. The dataframe is kept intact and the engineered columns are appended to the full model table.

In [30]:
import numpy as np
import pandas as pd
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq


def resolve_project_dirs(start: Path | None = None):
    """Resolve the assignment and project roots from any notebook launch location."""
    start = (start or Path.cwd()).resolve()
    for current in (start, *start.parents):
        if current.name == "Final Assignment" and (current / "telecom").is_dir() and (current / "data").is_dir():
            final_assignment_dir = current
            project_root = current.parent
            break
        if (current / "Final Assignment").is_dir() and (current / "docs").is_dir():
            project_root = current
            final_assignment_dir = current / "Final Assignment"
            break
    else:
        raise FileNotFoundError("Could not locate the project root from the current working directory.")
    docs_root = project_root / "docs"
    telecom_dir = final_assignment_dir / "telecom"
    intermediate_dir = final_assignment_dir / "data" / "intermediate"
    eda_reports_dir = docs_root / "4-EDA-reports"
    return project_root, final_assignment_dir, docs_root, telecom_dir, intermediate_dir, eda_reports_dir


if "telecom_dir" not in globals() or "intermediate_dir" not in globals() or "eda_reports_dir" not in globals():
    project_root, final_assignment_dir, docs_root, telecom_dir, intermediate_dir, eda_reports_dir = resolve_project_dirs()

if "df_clean" not in globals():
    client = pd.read_csv(telecom_dir / "Client.csv")
    record = pd.read_csv(telecom_dir / "Record.csv")
    df_clean = record.merge(client, on="Customer_ID", how="inner").drop(columns=["Customer_ID"])

feature_engineering_df = df_clean.copy()
new_feature_specs = [
    ("usage_to_charge", "mou_Mean / totmrc_Mean", lambda df: pd.Series(np.where(df["totmrc_Mean"].notna() & (df["totmrc_Mean"] != 0), df["mou_Mean"] / df["totmrc_Mean"], np.nan), index=df.index)),
    ("revenue_per_usage", "rev_Mean / (mou_Mean + 1)", lambda df: df["rev_Mean"] / (df["mou_Mean"] + 1)),
    ("trend_gap", "avg3mou - avg6mou", lambda df: df["avg3mou"] - df["avg6mou"]),
    ("revenue_gap", "avg3rev - avg6rev", lambda df: df["avg3rev"] - df["avg6rev"]),
    ("service_issue_score", "drop_vce_Mean + blck_vce_Mean + drop_blk_Mean", lambda df: df[["drop_vce_Mean", "blck_vce_Mean", "drop_blk_Mean"]].sum(axis=1, min_count=1)),
    ("support_intensity", "custcare_Mean + ccrndmou_Mean + cc_mou_Mean", lambda df: df[["custcare_Mean", "ccrndmou_Mean", "cc_mou_Mean"]].sum(axis=1, min_count=1)),
]

summary_rows = []
for feature_name, formula, builder in new_feature_specs:
    feature_engineering_df[feature_name] = builder(feature_engineering_df)
    series = feature_engineering_df[feature_name]
    non_null = series.dropna()
    summary_rows.append({
        "feature": feature_name,
        "formula": formula,
        "null_count": int(series.isna().sum()),
        "null_pct": float(series.isna().mean() * 100),
        "mean": float(non_null.mean()) if not non_null.empty else np.nan,
        "std": float(non_null.std()) if not non_null.empty else np.nan,
        "min": float(non_null.min()) if not non_null.empty else np.nan,
        "25%": float(non_null.quantile(0.25)) if not non_null.empty else np.nan,
        "50%": float(non_null.median()) if not non_null.empty else np.nan,
        "75%": float(non_null.quantile(0.75)) if not non_null.empty else np.nan,
        "max": float(non_null.max()) if not non_null.empty else np.nan,
    })

feature_engineering_summary_df = pd.DataFrame(summary_rows).sort_values(
    by=["feature"], ascending=[True]
).reset_index(drop=True)

telecom_dir.mkdir(parents=True, exist_ok=True)
eda_reports_dir.mkdir(parents=True, exist_ok=True)
intermediate_dir.mkdir(parents=True, exist_ok=True)

report_path = eda_reports_dir / "feature_engineering_v1.md"
report_lines = [
    "# FEATURE ENGINEERING",
    "",
    "The feature engineering step adds six derived signals recommended by the EDA plan. The dataframe is kept intact and the new columns are appended to the full modeling table.",
    "",
    "## Derived Features",
    "",
    "- `usage_to_charge = mou_Mean / totmrc_Mean`",
    "- `revenue_per_usage = rev_Mean / (mou_Mean + 1)`",
    "- `trend_gap = avg3mou - avg6mou`",
    "- `revenue_gap = avg3rev - avg6rev`",
    "- `service_issue_score = drop_vce_Mean + blck_vce_Mean + drop_blk_Mean`",
    "- `support_intensity = custcare_Mean + ccrndmou_Mean + cc_mou_Mean`",
    "",
    "## Summary Table",
    "",
    feature_engineering_summary_df.to_string(index=False),
    "",
    "## Validation",
    f"- Engineered features added: {len(new_feature_specs)}",
    f"- Rows: {len(feature_engineering_df)}",
    f"- Columns before: {df_clean.shape[1]}",
    f"- Columns after: {feature_engineering_df.shape[1]}",
    "- Null counts and distribution statistics are recorded in the summary table below.",
    ]
report_path.write_text("\n".join(report_lines), encoding="utf-8")

csv_path = telecom_dir / "feature_engineering_summary.csv"
feature_engineering_summary_df.to_csv(csv_path, index=False)

parquet_path = intermediate_dir / "E7_feature_engineering.parquet"
pq.write_table(pa.Table.from_pandas(feature_engineering_df, preserve_index=False), parquet_path)


In [31]:
display_cols = ["feature", "null_count", "null_pct", "mean", "std", "min", "25%", "50%", "75%", "max"]
print(feature_engineering_summary_df[display_cols].to_string(index=False))
print("\nNull counts by engineered feature:")
print(feature_engineering_summary_df[["feature", "null_count"]].to_string(index=False))
print("feature_engineering_summary.csv created successfully:", (telecom_dir / "feature_engineering_summary.csv").exists())
print("E7_feature_engineering.parquet created successfully:", (intermediate_dir / "E7_feature_engineering.parquet").exists())


            feature  null_count  null_pct      mean        std         min        25%       50%       75%         max
        revenue_gap        2839     2.839  0.398771  19.146355  -774.00000  -3.000000  0.000000  3.000000  738.000000
  revenue_per_usage         357     0.357  0.745688   4.132132    -6.16750   0.089803  0.140197  0.257510  163.230000
service_issue_score           0     0.000 20.022300  30.580090     0.00000   3.333333 10.666667 24.666667  840.000000
  support_intensity           0     0.000 10.141046  27.620691     0.00000   0.000000  0.000000  9.000000 2139.616667
          trend_gap        2839     2.839  6.577824 145.314008 -2528.00000 -37.000000  2.000000 50.000000 2284.000000
    usage_to_charge         686     0.686 11.338052  30.390763 -6430.30303   4.168056  8.558361 14.677330  833.000000

Null counts by engineered feature:
            feature  null_count
        revenue_gap        2839
  revenue_per_usage         357
service_issue_score           0
  support_